# Robust moving horizon estimation with GNSS outliers

Companion notebook for

> N. N. Deniz, G. M. Sanchez, F. A. Auat Cheein and L. L. Giovanini, "Robust Moving Horizon Estimation for Autonomous Agricultural Vehicles With GNSS Outliers Using a Robust Loss Function", *IEEE Robotics and Automation Letters*, 10(10):10815-10820, 2025. [doi:10.1109/LRA.2025.3606377](https://doi.org/10.1109/LRA.2025.3606377)

Run the cells top to bottom. **The numbers here are not the paper's numbers** - the scenario, the speeds and the tuning differ, and this problem is easier than the paper's. `NOTES.md` in the repository says exactly how.

## 1. Setup

The vehicle model, the path machinery and the controller are not re-implemented here: they come from the `nmppfc` repository.

In [ ]:
import os, subprocess, sys

if not os.path.exists('mhe-outlier-robust-v1'):
    subprocess.run(['git','clone','-q','https://github.com/nahueldeniz/mhe-outlier-robust-v1'], check=True)
if not os.path.exists('nmppfc'):
    subprocess.run(['git','clone','-q','https://github.com/nahueldeniz/nmppfc'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','casadi','pyyaml'], check=True)

sys.path.insert(0, 'mhe-outlier-robust-v1')
os.environ['NMPPFC_PATH'] = os.path.abspath('nmppfc')
print('ready')

## 2. The loss, and why alpha = 1

`psi` is Eq. (3): the square of the derivative of the General Adaptive Robust Loss. Its tail goes as $|x|^{2\alpha-2}$, so at $\alpha = 1$ the cost **saturates** at $1/c^2$ and its derivative - what actually drags the estimate - goes to zero. At $\alpha \to 2$ it is the quadratic the paper replaces.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from mhe_outlier_robust import loss

x = np.linspace(0, 40, 800)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a in (1.0, 1.5, 1.9, 1.99):
    ax[0].plot(x, np.clip(loss.psi(x, a), 0, 3), label=f'alpha = {a}')
ax[0].axhline(1.0, color='0.8', lw=1)
ax[0].set_xlabel('residual [m]'); ax[0].set_ylabel('stage cost'); ax[0].set_ylim(0, 3)
ax[0].legend(); ax[0].set_title('the cost stops growing at alpha = 1')

h = 1e-4
for a in (1.0, 1.5, 1.9):
    slope = (loss.psi(x + h, a) - loss.psi(x - h, a)) / (2 * h)
    ax[1].semilogy(x[1:], np.abs(slope[1:]) + 1e-16, label=f'alpha = {a}')
ax[1].set_xlabel('residual [m]'); ax[1].set_ylabel('|d psi / dx|  (the pull on the estimate)')
ax[1].legend(); ax[1].set_title('and stops pulling')
plt.tight_layout(); plt.show()

print('saturation at alpha=1, c=1:', loss.saturation(1.0, 1.0))
print('psi(1000 m, alpha=1)      :', float(loss.psi(1e3, 1.0)))

## 3. The path and the vehicle

The path is Fig. 2 rebuilt from its geometry: five 20 m rows 1.5 m apart, joined by bulb turns of radius 2 m. The state is the paper's redundant vector `q`, and the chain equalities are hard constraints inside the estimator.

In [ ]:
from mhe_outlier_robust import RedundantNTrailer, VehicleGeometry
from mhe_outlier_robust import path as paths

geo = VehicleGeometry(n_trailers=1, hitch=(0.342,), link=(1.08,),
                      body_length=(0.99, 1.0), body_width=(0.67, 0.8))
model = RedundantNTrailer(geo, ts=0.05)
path = paths.agricultural()

pts = paths.polyline()
plt.figure(figsize=(9, 3.2)); plt.plot(pts[:, 0], pts[:, 1], lw=2)
plt.gca().set_aspect('equal'); plt.xlabel('x [m]'); plt.ylabel('y [m]')
plt.title(f'path of Fig. 2, {paths.length():.1f} m'); plt.show()

print('q has', model.nq, 'entries; the output has', model.ny)
print('outlier-affected output rows (the tractor coordinates):', model.outlier_rows)

## 4. The closed loop

Sensors -> robust MHE -> reference -> controller -> plant, with a quarter of the GNSS samples corrupted. The controller is `nmppfc.NMPPFC`, handed the estimator's **full** state vector.

In [ ]:
from mhe_outlier_robust import EstimatorConfig, sim
from mhe_outlier_robust.sensors import SensorModel

cfg = EstimatorConfig(n_window=10, ts=0.05, alpha=1.0, c=1.0)
sensors = SensorModel(n_trailers=1, outlier_rows=model.outlier_rows,
                      n_window=cfg.n_window, mode='scattered', delta=0.25, seed=7)
res = sim.run(model, path, est_cfg=cfg, sensors=sensors, steps=400, speed=1.0)
for k, v in res.metrics.items():
    print(f'{k:24s} {v}')

In [ ]:
hit = res.corrupted.astype(bool)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(res.t, res.measurements[:, -2], '.', ms=3, color='0.75', label='measured x')
ax[0].plot(res.t[hit], res.measurements[hit, -2], '.', ms=5, color='tab:red', label='corrupted')
ax[0].plot(res.t, res.states[:, -4], lw=2, label='true x')
ax[0].plot(res.t, res.estimates[:, -4], lw=1.2, ls='--', label='estimated x')
ax[0].set_xlabel('t [s]'); ax[0].set_ylabel('x [m]'); ax[0].legend(fontsize=8)

ax[1].plot(res.t, res.est_error, label='estimation error [m]')
ax[1].plot(res.t, res.deviation, label='deviation from the path [m]')
ax[1].plot(res.t, res.rtf, label='real-time factor')
ax[1].axhline(1.0, color='0.8', lw=1); ax[1].set_xlabel('t [s]'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 5. Against the quadratic it replaces

The same estimator with `alpha -> 2` and the residual box opened. **This is our baseline, not any of the five methods the paper compares against** - those were implemented and tuned by the authors from their own references and are not in this repository.

In [ ]:
rows = []
for label, c in (('proposed (alpha = 1)', cfg), ('our quadratic baseline', sim.l2_baseline_config(cfg))):
    for delta in (0.0, 0.10, 0.25, 0.50):
        s = SensorModel(n_trailers=1, outlier_rows=model.outlier_rows, n_window=c.n_window,
                        mode='clean' if delta == 0 else 'scattered', delta=delta, seed=7)
        r = sim.run(model, path, est_cfg=c, sensors=s, steps=280, speed=1.0)
        rows.append((label, delta, r.metrics['mean_est_error_m'], r.metrics['mean_deviation_m']))
        print(f'{label:24s} delta={delta:4.0%}  est {rows[-1][2]:.3f} m  dev {rows[-1][3]:.3f} m')

In [ ]:
import numpy as np
plt.figure(figsize=(7, 4))
for label in dict.fromkeys(r[0] for r in rows):
    sub = [r for r in rows if r[0] == label]
    plt.semilogy([s[1] for s in sub], [s[2] for s in sub], 'o-', label=label)
plt.xlabel('outlier probability'); plt.ylabel('mean estimation error [m]')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 6. Sweep the tuning

`alpha` and `c` are parameters of the NLP, so this is a loop, not a rebuild. The paper's own sweep (its Fig. 3a, 3b) is five runs per cell - the columns are readable, the neighbouring cells are not, and the same caution applies here.

In [ ]:
from dataclasses import replace

for alpha in (1.0, 1.3, 1.6, 1.9):
    for cc in (0.1, 1.0, 10.0):
        s = SensorModel(n_trailers=1, outlier_rows=model.outlier_rows, n_window=10,
                        mode='scattered', delta=0.25, seed=7)
        r = sim.run(model, path, est_cfg=replace(cfg, alpha=alpha, c=cc), sensors=s,
                    steps=200, speed=1.0)
        print(f'alpha={alpha:4.1f}  c={cc:5.1f}  est {r.metrics["mean_est_error_m"]:.3f} m  '
              f'RTF {r.metrics["mean_rtf"]:.2f}')

## 7. Read the notes

`NOTES.md` records every deviation from the paper: the fixed arrival cost, the wrapped angular residuals, the reconstructed path, the sampling rate this path needs, and what the numbers above do and do not mean. Read it before quoting anything from this notebook.